# jev-local Colabランチャー

このノートブックはリポジトリをcloneして依存をインストールするだけ。
実際のロジックは全てリポジトリ側のモジュールにある。
生成物・チェックポイントはGoogle DriveかHF Hubに保存すること。

In [ ]:
!git clone https://github.com/fukayatti/jev-japanese-judgment.git
%cd jev-japanese-judgment
!pip install -q -r requirements.txt
!pip install -q vllm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/jev-japanese-judgment-checkpoints'

In [ ]:
# Colabのユーザーシークレット(左メニューの鍵アイコン)に HF_TOKEN を登録しておくこと
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
import subprocess
from pathlib import Path

from data.convert import chabsa, jcommonsenseqa, jsnli

# JCommonsenseQA / chABSA は HF Hub (parquet) から直接ロードできる
jcqa_examples = jcommonsenseqa.convert('train')
chabsa_examples = chabsa.convert('train')

# JSNLIは配布形式がzipなのでダウンロード・展開してからパスを渡す
jsnli_dir = Path('data/raw/jsnli')
jsnli_dir.mkdir(parents=True, exist_ok=True)
if not (jsnli_dir / 'jsnli_1.1' / 'train_w_filtering.tsv').exists():
    subprocess.run(['curl', '-sL', '-o', str(jsnli_dir / 'jsnli.zip'),
                     'https://nlp.ist.i.kyoto-u.ac.jp/nl-resource/JSNLI/jsnli_1.1.zip'], check=True)
    subprocess.run(['unzip', '-o', '-q', str(jsnli_dir / 'jsnli.zip'), '-d', str(jsnli_dir)], check=True)

jsnli_examples = jsnli.convert(jsnli_dir / 'jsnli_1.1' / 'train_w_filtering.tsv')

all_examples = jcqa_examples + chabsa_examples + jsnli_examples
print(f'jcqa={len(jcqa_examples)} chabsa={len(chabsa_examples)} jsnli={len(jsnli_examples)} total={len(all_examples)}')

In [ ]:
from data.augment.generate import build_prompts, run_batch_generation

# まずvLLMのguided_decoding APIがインストール済みバージョンで動くか、
# 少数サンプルで確認すること(vLLMのバージョンでAPI仕様が変わることがある)
jobs = build_prompts(all_examples[:1000])
results = run_batch_generation(jobs)

In [ ]:
from train import main as train_main

model = train_main(all_examples)

In [ ]:
from scripts.push_to_hub import push

# HF_TOKENは前段のセルで環境変数にセット済み。ここではrepo_idのみ指定する
push(all_examples, repo_id="fukayatti/jev-japanese-judgment", private=False)